# Encoder-Decoder মডেল: T5 এবং BART

তিনটি অংশ:
  1. T5-স্টাইল span corruption -- একটানা span-গুলোকে sentinel token দিয়ে প্রতিস্থাপন, target-এ শুধু হারানো অংশ।
  2. BART-স্টাইল denoising -- চারটি corruption কৌশল (token masking, token deletion, sentence permutation, document rotation), target সর্বদা সম্পূর্ণ মূল text।
  3. README-এর ডায়াগ্রামের প্রকৃত encoder-decoder architecture -- cross-attention দিয়ে যুক্ত একটি bidirectional encoder স্ট্যাক এবং একটি causal decoder স্ট্যাক -- স্ক্র্যাচ থেকে তৈরি এবং Part 1-এ তৈরি বাস্তব T5-স্টাইল span-corruption জোড়ায় end to end training, তারপর (একটি held-out বাক্যের হারানো span ভরাট করতে) generate করা।

Runtime: CPU-তে ~1-2 মিনিট (Part 3 1200 step-এর জন্য training করে)।

Notebook-এ চালাতে: প্রতিটি কোষ উপরে থেকে নিচে চালান (Shift+Enter)।

In [ ]:
import math
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

random.seed(0)
torch.manual_seed(0)

## 1. T5-স্টাইল span corruption

নিচের কোষটি `t5_span_corruption` ফাংশন সংজ্ঞায়িত করে এবং `t5_demo()` চালিয়ে কয়েকটি উদাহরণে এর কাজ দেখায়।

In [ ]:
# ---------------------------------------------------------------------------
# 1. T5-স্টাইল span corruption
# ---------------------------------------------------------------------------

def t5_span_corruption(tokens, noise_density=0.15, max_span_length=3):
    n = len(tokens)
    target_noise = max(1, round(n * noise_density))
    corrupted = [False] * n
    noise_so_far = 0

    attempts = 0
    while noise_so_far < target_noise and attempts < 100:
        attempts += 1
        start = random.randrange(n)
        span_len = random.randint(1, max_span_length)
        span_len = min(span_len, n - start, target_noise - noise_so_far)
        if span_len <= 0 or any(corrupted[start:start + span_len]):
            continue   # ইতিমধ্যে বাছাই করা span-এর সাথে ওভারল্যাপ, বা খাপ খায় না -- আবার চেষ্টা
        for j in range(start, start + span_len):
            corrupted[j] = True
        noise_so_far += span_len

    input_tokens, target_tokens = [], []
    sentinel_id = 0
    i = 0
    while i < n:
        if corrupted[i]:
            sentinel = f"<extra_id_{sentinel_id}>"
            input_tokens.append(sentinel)
            target_tokens.append(sentinel)
            while i < n and corrupted[i]:
                target_tokens.append(tokens[i])
                i += 1
            sentinel_id += 1
        else:
            input_tokens.append(tokens[i])
            i += 1
    target_tokens.append(f"<extra_id_{sentinel_id}>")  # T5 একটি চূড়ান্ত sentinel দিয়ে সমাপ্ত হয়
    return input_tokens, target_tokens


def t5_demo():
    print("=" * 70)
    print("1. T5-STYLE SPAN CORRUPTION")
    print("=" * 70)
    sentences = [
        "the quick brown fox jumps over the lazy dog",
        "the sun rises over the dark forest every morning",
    ]
    for sentence in sentences:
        tokens = sentence.split()
        corrupted_input, target = t5_span_corruption(tokens)
        print(f"  original: {' '.join(tokens)}")
        print(f"  input:    {' '.join(corrupted_input)}")
        print(f"  target:   {' '.join(target)}\n")
    print("-> The target is much SHORTER than the input -- it only reconstructs")
    print("   the missing spans, tagged by sentinel, not the whole sentence.")


t5_demo()

## 2. BART-স্টাইল denoising corruption

নিচের কোষটি চারটি BART-style corruption ফাংশন সংজ্ঞায়িত করে এবং `bart_demo()` চালিয়ে প্রতিটির input/target দেখায়।

In [ ]:
# ---------------------------------------------------------------------------
# 2. BART-স্টাইল denoising corruption
# ---------------------------------------------------------------------------

def bart_token_masking(tokens, mask_prob=0.3):
    corrupted = [("[MASK]" if random.random() < mask_prob else t) for t in tokens]
    return corrupted, list(tokens)   # target = সম্পূর্ণ মূল sequence


def bart_token_deletion(tokens, delete_prob=0.3):
    corrupted = [t for t in tokens if random.random() >= delete_prob]
    return corrupted, list(tokens)


def bart_sentence_permutation(sentences):
    shuffled = sentences[:]
    # সত্যিই ভিন্ন না হওয়া পর্যন্ত আবার এলোমেলো করো -- মাত্র কয়েকটি বাক্যের
    # document-এ random.shuffle সত্যিই মূল ক্রমে ফিরে আসতে পারে; আমরা কেবল
    # নিশ্চিত হতে retry করি যে এই demo সবসময় একটি বাস্তব পরিবর্তন দেখায়।
    for _ in range(20):
        random.shuffle(shuffled)
        if shuffled != sentences or len(sentences) < 2:
            break
    return shuffled, sentences   # target = সঠিক মূল ক্রম


def bart_document_rotation(tokens):
    if len(tokens) < 2:
        return list(tokens), list(tokens)
    pivot = random.randint(1, len(tokens) - 1)
    rotated = tokens[pivot:] + tokens[:pivot]
    return rotated, list(tokens)   # target = প্রকৃত মূল ক্রম/শুরু


def bart_demo():
    print("\n" + "=" * 70)
    print("2. BART-STYLE DENOISING (target is always the FULL original text)")
    print("=" * 70)

    tokens = "the quick brown fox jumps over the lazy dog".split()

    corrupted, target = bart_token_masking(tokens)
    print("Token masking:")
    print(f"  input:  {' '.join(corrupted)}")
    print(f"  target: {' '.join(target)}\n")

    corrupted, target = bart_token_deletion(tokens)
    print("Token deletion (model must infer WHERE tokens are missing, not just what):")
    print(f"  input:  {' '.join(corrupted)}")
    print(f"  target: {' '.join(target)}\n")

    sentences = [
        "the fox saw the hen.",
        "the hen ran into the barn.",
        "the fox followed close behind.",
    ]
    shuffled, original_order = bart_sentence_permutation(sentences)
    print("Sentence permutation:")
    print(f"  input:  {' '.join(shuffled)}")
    print(f"  target: {' '.join(original_order)}\n")

    corrupted, target = bart_document_rotation(tokens)
    print("Document rotation (model must identify the TRUE starting point):")
    print(f"  input:  {' '.join(corrupted)}")
    print(f"  target: {' '.join(target)}")

    print("\n-> Every BART target is the complete, correctly-ordered original text --")
    print("   a strictly harder reconstruction target than T5's 'just the missing")
    print("   pieces', which is exactly why BART tends to shine on generation-heavy")
    print("   tasks like summarization: it was always trained to produce full,")
    print("   fluent text, never just isolated spans.")


bart_demo()

## 3. প্রকৃত encoder-decoder architecture: তৈরি ও training

নিচের কোষটি README-এর ডায়াগ্রামের সম্পূর্ণ architecture তৈরি করে -- cross-attention দিয়ে যুক্ত encoder ও decoder স্ট্যাক -- Part 1-এর বাস্তব T5 span-corruption জোড়ায় end to end training করে, তারপর `architecture_demo()` চালিয়ে held-out বাক্যের হারানো span ভরাট করে।

In [ ]:
# ---------------------------------------------------------------------------
# 3. README-এর ডায়াগ্রামের প্রকৃত encoder-decoder architecture, স্ক্র্যাচ
# থেকে তৈরি এবং বাস্তব T5-স্টাইল span-corruption জোড়ায় (উপরের Part 1-এর
# t5_span_corruption-কে data pipeline হিসেবে ব্যবহার করে) end to end training
# করা।
# ---------------------------------------------------------------------------

CORPUS_SENTENCES = [
    "the quick brown fox jumps over the lazy dog",
    "the sun rises over the dark forest every morning",
    "the moon shines over the dark forest every night",
    "birds sing songs in the tall green trees",
    "the small dog barks at the tall cat",
    "the tall cat runs away from the small dog",
    "the lazy dog barks at the quick brown fox",
    "the quick brown fox runs into the dark forest",
]
MAX_SPAN_LENGTH = 2
NUM_SENTINELS = 4   # এই ছোট বাক্যগুলোতে MAX_SPAN_LENGTH=2 corruption-এর জন্য যথেষ্ট

SPECIALS = ["[PAD]", "[BOS]", "[EOS]"]
SENTINELS = [f"<extra_id_{i}>" for i in range(NUM_SENTINELS)]
WORDS = sorted({w for s in CORPUS_SENTENCES for w in s.split()})
VOCAB = SPECIALS + SENTINELS + WORDS
STOI = {w: i for i, w in enumerate(VOCAB)}
ITOS = {i: w for i, w in enumerate(VOCAB)}
VOCAB_SIZE = len(VOCAB)
PAD_ID, BOS_ID, EOS_ID = (STOI[t] for t in SPECIALS)

MAX_SRC_LEN = 12
MAX_TGT_LEN = 10


def encode_padded(word_tokens, max_len, append_eos=False):
    ids = [STOI[w] for w in word_tokens]
    if append_eos:
        ids = ids + [EOS_ID]
    ids = ids[:max_len]
    ids = ids + [PAD_ID] * (max_len - len(ids))
    return ids


def make_training_pair():
    """একটি (encoder_input, decoder_input, decoder_target) ট্রিপল, একটি এলোমেলো
    বাক্যে Part 1-এর বাস্তব T5 span-corruption ফাংশন চালিয়ে তৈরি।"""
    tokens = random.choice(CORPUS_SENTENCES).split()
    input_tokens, target_tokens = t5_span_corruption(
        tokens, noise_density=0.3, max_span_length=MAX_SPAN_LENGTH
    )
    encoder_input = encode_padded(input_tokens, MAX_SRC_LEN)
    # আদর্শ teacher-forcing সেটআপ: decoder INPUT হলো target-টি right-শিফট করা
    # ([BOS] দিয়ে prefixed); decoder TARGET হলো target-টি নিজেই, [EOS] দিয়ে
    # শেষ হওয়া, যাতে মডেল কখন generate বন্ধ করতে হয় সেটাও শেখে।
    decoder_input = encode_padded([("[BOS]")] + target_tokens, MAX_TGT_LEN)
    decoder_target = encode_padded(target_tokens, MAX_TGT_LEN, append_eos=True)
    return encoder_input, decoder_input, decoder_target


def get_training_batch(batch_size):
    triples = [make_training_pair() for _ in range(batch_size)]
    enc_in, dec_in, dec_tgt = zip(*triples)
    return (
        torch.tensor(enc_in, dtype=torch.long),
        torch.tensor(dec_in, dtype=torch.long),
        torch.tensor(dec_tgt, dtype=torch.long),
    )


class MultiHeadAttention(nn.Module):
    """এই কোর্সজুড়ে ব্যবহৃত একই mechanism -- query sequence key/value
    sequence থেকে ভিন্ন হতে পারে এমনভাবে generalization করা। এজন্যই একটি
    মাত্র class encoder self-attention, decoder self-attention এবং decoder-এর
    cross-attention (README ডায়াগ্রামের সেই বক্স যেখানে Q decoder থেকে আসে
    কিন্তু K/V encoder-এর output থেকে আসে) তিনটিই হিসেবে কাজ করতে পারে।"""

    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def _split_heads(self, x):
        batch, T, d_model = x.shape
        return x.view(batch, T, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, query_input, key_value_input, mask=None):
        batch, T_q, d_model = query_input.shape
        Q = self._split_heads(self.W_q(query_input))
        K = self._split_heads(self.W_k(key_value_input))
        V = self._split_heads(self.W_v(key_value_input))

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        out = (weights @ V).transpose(1, 2).contiguous().view(batch, T_q, d_model)
        return self.W_o(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x, src_pad_mask):
        normed = self.ln1(x)
        x = x + self.self_attn(normed, normed, mask=src_pad_mask)  # bidirectional self-attention
        x = x + self.ffn(self.ln2(x))
        return x


class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.ln3 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x, encoder_output, self_mask, cross_mask):
        normed = self.ln1(x)
        x = x + self.self_attn(normed, normed, mask=self_mask)          # causal self-attention
        # Cross-attention: query DECODER থেকে, key/value ENCODER থেকে --
        # README ডায়াগ্রামের সেই বক্স যার Lessons 1-2-এ কোনো সমতুল্য নেই।
        x = x + self.cross_attn(self.ln2(x), encoder_output, mask=cross_mask)
        x = x + self.ffn(self.ln3(x))
        return x


class EncoderDecoderModel(nn.Module):
    """README ডায়াগ্রামের সম্পূর্ণ architecture: একটি encoder স্ট্যাক
    (bidirectional self-attention) এবং একটি decoder স্ট্যাক (causal
    self-attention + encoder-এর output-এ cross-attention), একটি মাত্র
    vocabulary ভাগ করে -- হুবহু T5-এর সেটআপ, যেখানে source এবং target একই
    text-to-text vocabulary থেকে আঁকা হয়।"""

    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_len):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)
        self.encoder_blocks = nn.ModuleList(
            [EncoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)]
        )
        self.decoder_blocks = nn.ModuleList(
            [DecoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)]
        )
        self.final_norm = nn.LayerNorm(d_model)
        self.output_head = nn.Linear(d_model, vocab_size)

    def embed(self, token_ids):
        T = token_ids.shape[1]
        positions = torch.arange(T, device=token_ids.device)
        return self.token_embedding(token_ids) + self.position_embedding(positions)

    def encode(self, src_ids):
        src_pad_mask = (src_ids != PAD_ID).view(src_ids.shape[0], 1, 1, -1)  # (batch,1,1,src_len)
        x = self.embed(src_ids)
        for block in self.encoder_blocks:
            x = block(x, src_pad_mask)
        return x, src_pad_mask

    def decode(self, tgt_ids, encoder_output, src_pad_mask):
        batch, T = tgt_ids.shape
        tgt_pad_mask = (tgt_ids != PAD_ID).view(batch, 1, 1, T)          # (batch,1,1,T)
        causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=tgt_ids.device))
        self_mask = causal.view(1, 1, T, T) & tgt_pad_mask               # causal এবং non-padding
        x = self.embed(tgt_ids)
        for block in self.decoder_blocks:
            x = block(x, encoder_output, self_mask, src_pad_mask)
        x = self.final_norm(x)
        return self.output_head(x)   # (batch, T, vocab_size)

    def forward(self, src_ids, tgt_ids):
        encoder_output, src_pad_mask = self.encode(src_ids)
        return self.decode(tgt_ids, encoder_output, src_pad_mask)

    @torch.no_grad()
    def generate(self, src_ids, max_new_tokens):
        encoder_output, src_pad_mask = self.encode(src_ids)
        generated = torch.full((src_ids.shape[0], 1), BOS_ID, dtype=torch.long)
        for _ in range(max_new_tokens):
            logits = self.decode(generated, encoder_output, src_pad_mask)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)
            if (next_token == EOS_ID).all():
                break
        return generated


def architecture_demo():
    print("\n" + "=" * 70)
    print("3. THE ACTUAL ENCODER-DECODER ARCHITECTURE, BUILT AND TRAINED")
    print("=" * 70)
    print(f"Shared vocabulary ({VOCAB_SIZE} tokens): specials + {NUM_SENTINELS} sentinels "
          f"+ {len(WORDS)} words")

    model = EncoderDecoderModel(
        VOCAB_SIZE, d_model=64, num_heads=4, d_ff=256, num_layers=2,
        max_len=max(MAX_SRC_LEN, MAX_TGT_LEN),
    )
    print(f"Model parameter count: {sum(p.numel() for p in model.parameters()):,}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
    print("\nTraining on real T5 span-corruption pairs (loss ignores [PAD] positions)...")
    for step in range(1, 1201):
        enc_in, dec_in, dec_tgt = get_training_batch(batch_size=32)
        logits = model(enc_in, dec_in)
        loss = F.cross_entropy(
            logits.view(-1, VOCAB_SIZE), dec_tgt.view(-1), ignore_index=PAD_ID
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 300 == 0 or step == 1:
            print(f"  step {step:5d}  loss = {loss.item():.4f}")

    print("\nFilling in missing spans on held-out sentences (greedy decoding):")
    model.eval()
    for sentence in CORPUS_SENTENCES[:3]:
        tokens = sentence.split()
        input_tokens, true_target = t5_span_corruption(
            tokens, noise_density=0.3, max_span_length=MAX_SPAN_LENGTH
        )
        src_ids = torch.tensor([encode_padded(input_tokens, MAX_SRC_LEN)])
        generated = model.generate(src_ids, max_new_tokens=MAX_TGT_LEN)[0].tolist()
        predicted = [ITOS[i] for i in generated[1:] if i not in (PAD_ID, EOS_ID)]

        print(f"  input:     {' '.join(input_tokens)}")
        print(f"  true:      {' '.join(true_target)}")
        print(f"  predicted: {' '.join(predicted)}\n")

    print("-> The encoder saw the corrupted sentence with NO causal mask (bidirectional --")
    print("   same as Lesson 2's BERT); the decoder generated the missing spans one")
    print("   sentinel-tagged token at a time, using CAUSAL self-attention over what")
    print("   it has produced so far PLUS cross-attention back into the encoder's full,")
    print("   bidirectionally-processed output. That combination -- bidirectional input")
    print("   understanding + autoregressive output generation -- is exactly what")
    print("   neither Lesson 1 (decoder-only) nor Lesson 2 (encoder-only) can do alone.")


architecture_demo()

In [ ]:
def main():
    t5_demo()
    bart_demo()
    architecture_demo()


main()